In [0]:
from databricks.sdk import WorkspaceClient
wc = WorkspaceClient()
user_name = wc.current_user.me().display_name

user_name_cleaned = ''.join(['_' if not c.isalpha() else c for c in user_name.lower()])
display(user_name_cleaned)

root_path = f"/Workspace/Users/{user_name}/workshop/streaming_etl/2_declarative_pipeline/flights_streaming_pipeline"

includes = f"{root_path}/transformations/**"

schema = "flights_streaming_pipeline_dev"

catalog = f"{user_name_cleaned}_catalog"

# event_log_payload = {
#     "name": "eventlog_flights_streaming_dlt_dev",
#     "schema": "flights_streaming_dlt_dev",
#     "catalog": f"{user_name_cleaned}_catalog"
#   }

pipeline_payload = {
  "name": "flights_streaming_dlt_DEV",
  "libraries": [
    {
      "glob": {
        "include": includes
      }
    }
  ],
  "schema": schema,
  "development": "true",
  "photon": "true",
  "channel": "CURRENT",
  "catalog": catalog,
  "serverless": "true",
  # "event_log": event_log_payload,
  "root_path": root_path
}

# Create the DLT pipeline using the workspace client
try:
    pipeline = wc.pipelines.create(**pipeline_payload)
    print(f"Pipeline created successfully!")
    print(f"Pipeline ID: {pipeline.pipeline_id}")
    print(f"Pipeline Name: {pipeline.name}")
    print(f"Pipeline URL: https://{wc.config.host}/#joblist/pipelines/{pipeline.pipeline_id}")
except Exception as e:
    print(f"Error creating pipeline: {str(e)}")
    # If pipeline already exists, try to find it
    try:
        existing_pipelines = wc.pipelines.list()
        for p in existing_pipelines:
            if p.name == pipeline_payload["name"]:
                print(f"Pipeline '{p.name}' already exists with ID: {p.pipeline_id}")
                print(f"Pipeline URL: https://{wc.config.host}/#joblist/pipelines/{p.pipeline_id}")
                break
    except Exception as list_error:
        print(f"Error listing pipelines: {str(list_error)}")